In [32]:
#Cell 1 : Imports & Load Dataset
from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE_DIR      = Path.cwd().parent
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

raw_path = RAW_DIR / "paysim.csv"
df = pd.read_csv(raw_path)

print(df.shape)
print(df.head(3))



(6362620, 11)
   step      type   amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT  9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT  1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER   181.00  C1305486145          181.0            0.00   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  


In [33]:
#Cell 2: Load EDA Summary Hints
summary_path = PROCESSED_DIR / "eda_summary.json"

with summary_path.open("r") as f:
    eda_summary = json.load(f)

feature_hints = eda_summary["feature_engineering_hints"]

for i, hint in enumerate(feature_hints, start=1):
    print(f"{i}. {hint}")


1. is_transfer_or_cashout — only these types have fraud
2. is_zero_newbalanceOrig — 98% of fraud drains origin to 0
3. amount_log — amount is heavily right-skewed, log-transform needed
4. step_mod_24 — hour-of-day signal from step column
5. is_late_period — Late period has 10x higher fraud rate
6. orig_balance_diff — deviation from expected balance equation
7. dest_balance_diff — deviation from expected balance equation


In [34]:
#Cell 3: Feature Engineering (1,2,3)
df["is_transfer_or_cashout"] = df["type"].isin(["TRANSFER", "CASH_OUT"]).astype(int)

df["is_zero_newbalanceOrig"] = (df["newbalanceOrig"] == 0).astype(int)

df["amount_log"] = np.log1p(df["amount"])

print(df[["type", "amount", "newbalanceOrig",
          "is_transfer_or_cashout", "is_zero_newbalanceOrig", "amount_log"]].head(10))
print(f"\nis_transfer_or_cashout value counts:\n{df['is_transfer_or_cashout'].value_counts()}")
print(f"\nis_zero_newbalanceOrig value counts:\n{df['is_zero_newbalanceOrig'].value_counts()}")


       type    amount  newbalanceOrig  is_transfer_or_cashout  \
0   PAYMENT   9839.64       160296.36                       0   
1   PAYMENT   1864.28        19384.72                       0   
2  TRANSFER    181.00            0.00                       1   
3  CASH_OUT    181.00            0.00                       1   
4   PAYMENT  11668.14        29885.86                       0   
5   PAYMENT   7817.71        46042.29                       0   
6   PAYMENT   7107.77       176087.23                       0   
7   PAYMENT   7861.64       168225.59                       0   
8   PAYMENT   4024.36            0.00                       0   
9     DEBIT   5337.77        36382.23                       0   

   is_zero_newbalanceOrig  amount_log  
0                       0    9.194276  
1                       0    7.531166  
2                       1    5.204007  
3                       1    5.204007  
4                       0    9.364703  
5                       0    8.964275  
6   

In [35]:
#Cell 4: Feature Engineering (4,5)
df["step_mod_24"] = df["step"] % 24

df["is_late_period"] = (df["step"] > 494).astype(int)

print(df[["step", "step_mod_24", "is_late_period"]].head(10))
print(f"\nstep_mod_24 unique values (should be 0–23): {sorted(df['step_mod_24'].unique())}")
print(f"\nis_late_period value counts:\n{df['is_late_period'].value_counts()}")
print(f"\nFraud rate in late period: {df[df['is_late_period']==1]['isFraud'].mean():.4f}")
print(f"Fraud rate in early/mid period: {df[df['is_late_period']==0]['isFraud'].mean():.4f}")



   step  step_mod_24  is_late_period
0     1            1               0
1     1            1               0
2     1            1               0
3     1            1               0
4     1            1               0
5     1            1               0
6     1            1               0
7     1            1               0
8     1            1               0
9     1            1               0

step_mod_24 unique values (should be 0–23): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]

is_late_period value counts:
is_late_period
0    6051528
1     311092
Name: count, dtype: int64

Fraud rate in late period: 0.0087
Fraud rate in early/mid period: 0.0009


In [36]:
#Cell 5: Feature Engineering (6,7)
df["orig_balance_diff"] = (df["oldbalanceOrg"] - df["amount"]) - df["newbalanceOrig"]

df["dest_balance_diff"] = (df["oldbalanceDest"] + df["amount"]) - df["newbalanceDest"]

print(df[["oldbalanceOrg", "amount", "newbalanceOrig", "orig_balance_diff",
          "oldbalanceDest", "newbalanceDest", "dest_balance_diff", "isFraud"]].head(10))
print(f"\norig_balance_diff — fraud mean:  {df[df['isFraud']==1]['orig_balance_diff'].mean():.2f}")
print(f"orig_balance_diff — legit mean:  {df[df['isFraud']==0]['orig_balance_diff'].mean():.2f}")
print(f"\ndest_balance_diff — fraud mean:  {df[df['isFraud']==1]['dest_balance_diff'].mean():.2f}")
print(f"dest_balance_diff — legit mean:  {df[df['isFraud']==0]['dest_balance_diff'].mean():.2f}")


   oldbalanceOrg    amount  newbalanceOrig  orig_balance_diff  oldbalanceDest  \
0      170136.00   9839.64       160296.36       0.000000e+00             0.0   
1       21249.00   1864.28        19384.72       0.000000e+00             0.0   
2         181.00    181.00            0.00       0.000000e+00             0.0   
3         181.00    181.00            0.00       0.000000e+00         21182.0   
4       41554.00  11668.14        29885.86       0.000000e+00             0.0   
5       53860.00   7817.71        46042.29       0.000000e+00             0.0   
6      183195.00   7107.77       176087.23       0.000000e+00             0.0   
7      176087.23   7861.64       168225.59       0.000000e+00             0.0   
8        2671.00   4024.36            0.00      -1.353360e+03             0.0   
9       41720.00   5337.77        36382.23      -7.275958e-12         41898.0   

   newbalanceDest  dest_balance_diff  isFraud  
0            0.00            9839.64        0  
1           

In [37]:
#Cell 6 : One Hot Encoding type Column
type_dummies = pd.get_dummies(df["type"], prefix="type", dtype=int)

df = pd.concat([df, type_dummies], axis=1)

df = df.drop(columns=["type"])

print(df.shape)
print(f"\nNew type columns: {[col for col in df.columns if col.startswith('type_')]}")
print(f"\nSample of encoded columns:\n{df[[col for col in df.columns if col.startswith('type_')]].head(5)}")


(6362620, 22)

New type columns: ['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']

Sample of encoded columns:
   type_CASH_IN  type_CASH_OUT  type_DEBIT  type_PAYMENT  type_TRANSFER
0             0              0           0             1              0
1             0              0           0             1              0
2             0              0           0             0              1
3             0              1           0             0              0
4             0              0           0             1              0


In [38]:
#Cell 7 : Temporal train/test split
train_df = df[df["step"] <= 494].copy()
test_df  = df[df["step"] >  494].copy()

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"\nTrain fraud rate: {train_df['isFraud'].mean():.4f}")
print(f"Test fraud rate : {test_df['isFraud'].mean():.4f}")
print(f"\nTrain step range: {train_df['step'].min()} – {train_df['step'].max()}")
print(f"Test step range : {test_df['step'].min()} – {test_df['step'].max()}")


Train shape : (6051528, 22)
Test shape  : (311092, 22)

Train fraud rate: 0.0009
Test fraud rate : 0.0087

Train step range: 1 – 494
Test step range : 495 – 743


In [40]:
#Cell 8 : Valdate Feature Set & Save to Parquet
DROP_COLS = ["nameOrig", "nameDest", "isFlaggedFraud"]

train_df = train_df.drop(columns=DROP_COLS, errors="ignore")
test_df  = test_df.drop(columns=DROP_COLS, errors="ignore")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROCESSED_DIR / "train_features.parquet"
test_path  = PROCESSED_DIR / "test_features.parquet"

train_df.to_parquet(train_path, index=False, engine="fastparquet")
test_df.to_parquet(test_path,  index=False, engine="fastparquet")

print(f"Train saved : {train_path}")
print(f"Test saved  : {test_path}")
print(f"\nFinal train shape : {train_df.shape}")
print(f"Final test shape  : {test_df.shape}")
print(f"\nFinal columns ({len(train_df.columns)}):\n{list(train_df.columns)}")
print(f"\nAny nulls in train: {train_df.isnull().sum().sum()}")
print(f"Any nulls in test : {test_df.isnull().sum().sum()}")



Train saved : /home/minipekka/Hackathons/USM VHack 2026/nuroguard/data/processed/train_features.parquet
Test saved  : /home/minipekka/Hackathons/USM VHack 2026/nuroguard/data/processed/test_features.parquet

Final train shape : (6051528, 19)
Final test shape  : (311092, 19)

Final columns (19):
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'is_transfer_or_cashout', 'is_zero_newbalanceOrig', 'amount_log', 'step_mod_24', 'is_late_period', 'orig_balance_diff', 'dest_balance_diff', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']

Any nulls in train: 0
Any nulls in test : 0
